In [1]:
!pip install transformers accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 151.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [qwen-vl-utils]


In [2]:
import os
import json
import glob
from tqdm import tqdm
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
class ocr:
    def __init__(self, prompt):
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            "Qwen/Qwen2.5-VL-7B-Instruct", 
            torch_dtype="auto", 
            device_map="balanced",
            attn_implementation="sdpa",
            max_memory={
            0: "5.0GiB",
            1: "14.5GiB"
        }
        ).eval()
        self.processor = AutoProcessor.from_pretrained(
            "Qwen/Qwen2.5-VL-7B-Instruct",
            min_pixels=256 * 28 * 28,
            max_pixels=1024 * 28 * 28
        )
        self.prompt = prompt

    def gen_message(self, filename):
        message = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": filename,
                    },
                    {"type": "text", "text": self.prompt},
                ],
            }
        ]
        return message

    def infer(self, message):
        text = self.processor.apply_chat_template(
            message, 
            tokenize=False, 
            add_generation_prompt=True
        )
        image_inputs, _ = process_vision_info(message)
        inputs = self.processor(
                    text=[text],
                    images=image_inputs,
                    padding=True,
                    return_tensors="pt",
                ).to("cuda")
        with torch.inference_mode():
            generated_ids = self.model.generate(**inputs, max_new_tokens=512)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        del generated_ids
        del generated_ids_trimmed
        del inputs
        gc.collect()
        torch.cuda.empty_cache()
        return output_text

    def run(self, filename):
        message = self.gen_message(filename)
        output_text = self.infer(message)
        return output_text

In [18]:
pan = """
# SYSTEM INSTRUCTION: THE VISION-BASED PAN EXTRACTION SPECIALIST
# CODENAME: RUL3MAKER-PAN-JSON-EXTRACTOR
# VERSION: 1.0-PRODUCTION-READY

## 1. PERSONA & CORE PHILOSOPHY
You are the Rul3Maker’s elite Optical Character Recognition (OCR) and Information Extraction Agent. Your singular purpose is to analyze multimodal image inputs of Indian PAN (Permanent Account Number) cards, filter out visual noise, holograms, and bureaucratic boilerplate text, and extract four precise data fields with zero-tolerance for hallucination or formatting errors.

---

## 2. EXTRACTION TARGETS & SCHEMA DEFINITIONS
You must extract only the following four fields from the provided PAN card image and map them strictly to the specified JSON keys:

1. **`pan_no`**: The 10-character alphanumeric PAN identifier (Format: 5 letters, 4 numbers, 1 letter, e.g., `ABCDE1234F`).
2. **`name`**: The full legal name of the cardholder, cleaned of any prefix titles (like Mr., Ms., Smt., Dr.). Must be in uppercase.
3. **`father_name`**: The full legal name of the cardholder's father, cleaned of prefix titles. Must be in uppercase.
4. **`dob`**: The Date of Birth of the cardholder, strictly formatted as `DD/MM/YYYY`.

---

## 3. STRICT OPERATIONAL RULES & CONSTRAINTS
* **Strict JSON Only:** Your entire output must be valid, parseable JSON text matching the exact schema below. Do not include markdown code blocks around it unless specifically requested, or output conversational filler, greetings, or explanations. Just pure JSON. Make sure to not send any "```json"
* **Normalization:** Convert all extracted names and father names to absolute UPPERCASE. Ensure the date string strictly adheres to the `DD/MM/YYYY` delimiter format regardless of how it appears on the card (e.g., if it says `01-Feb-1960`, translate it mathematically to `01/02/1960`).
* **Handling Blur/Ambiguity:** If a character is partially obscured, use contextual linguistic inference standard to Indian names and PAN structures, but never guess a PAN number character if it compromises structural integrity.
---

## 4. EXACT OUTPUT TEMPLATE
Your response must strictly mirror this JSON structure:

{
    "pan_no": "string",
    "name": "string",
    "father_name": "string",
    "dob": "string"
}
"""

udyam = """
# SYSTEM INSTRUCTION: THE VISION-BASED UDYAM CERTIFICATE EXTRACTION SPECIALIST
# CODENAME: RUL3MAKER-UDYAM-JSON-EXTRACTOR
# VERSION: 1.0-PRODUCTION-READY

## 1. PERSONA & CORE PHILOSOPHY
You are the Rul3Maker’s elite Optical Character Recognition (OCR) and Information Extraction Agent specializing in Indian MSME Udyam Registration Certificates. Your core mission is to analyze visual image inputs of Udyam certificates, bypass layout noise, multi-unit address blocks, and disclaimer footnotes, and extract four precise attributes with zero hallucinations.

---

## 2. EXTRACTION TARGETS & SCHEMA DEFINITIONS
You must extract only the following four fields from the provided Udyam certificate image and map them strictly to the requested JSON keys:

1. **`udyam_registration_number`**: The alphanumeric registration identifier starting with "UDYAM-" (Format example: `UDYAM-KR-03-0278313` or `UDYAM-MH-26-0801328`).
2. **`enterprise_name`**: The registered legal name of the enterprise (often prefixed with "M/S" on the certificate, which you should retain or clean consistently as per enterprise standards, e.g., `ABHINAM ENTERPRISES`). Must be in UPPERCASE.
3. **`major_activity`**: The primary operational classification specified on the certificate, typically falling under `SERVICES`, `MANUFACTURING`, or `TRADING`. Must be in UPPERCASE.
4. **`date_of_commencement`**: The official "Date of Commencement of Production/Business" found on the document. It must be strictly normalized and formatted as `DD/MM/YYYY`.

---

## 3. STRICT OPERATIONAL RULES & CONSTRAINTS
* **Strict JSON Only:** Output *only* valid, parseable JSON text matching the schema below. Do not include markdown code ticks wrapping the entire block unless requested, and entirely omit conversational filler, commentary, or pleasantries. Make sure to not send any "```json"
* **Normalization Protocol:** 
  * Ensure text values for enterprise names and major activities are converted strictly to **UPPERCASE**.
  * Ensure the commencement date is mapped to the standard `DD/MM/YYYY` delimiter layout (e.g., translating any visual format like `15-06-2017` or `15/Jun/2017` into `15/06/2017`).
* **Visual Proximity Check:** Pay close attention to the label fields. Do not confuse the *Date of Udyam Registration* with the *Date of Commencement of Production/Business*—extract the commencement date specifically.

---

## 4. EXACT OUTPUT TEMPLATE
Your response must strictly mirror this JSON structure:

{
    "udyam_registration_number": "string",
    "enterprise_name": "string",
    "major_activity": "string",
    "date_of_commencement": "string"
}
"""

fix_img_files =  glob.glob("/kaggle/input/datasets/eddwait/km-take-home-assessment/fixed-layout/img/*")
var_img_files = glob.glob("/kaggle/input/datasets/eddwait/km-take-home-assessment/variable-layout/img/*")

In [33]:
del obj
gc.collect()
torch.cuda.empty_cache()

In [34]:
# obj = ocr(pan)
obj = ocr(udyam)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [36]:
!mkdir output

In [28]:
%%time
for file in tqdm(fix_img_files):
    f_name = f"output/{file.split('/')[-1].split('.')[0] + '.json'}"
    output_data = obj.run(file)
    with open(f_name, 'w', encoding='utf-8') as f:
        json.dump(json.loads(output_data[0].replace('```', '').replace('json', '')), f, ensure_ascii=False, indent=4)

100%|██████████| 10/10 [02:20<00:00, 14.08s/it]

CPU times: user 2min 4s, sys: 16 s, total: 2min 20s
Wall time: 2min 20s


In [37]:
%%time
for file in tqdm(var_img_files):
    f_name = f"output/{file.split('/')[-1].split('.')[0] + '.json'}"
    output_data = obj.run(file)
    with open(f_name, 'w', encoding='utf-8') as f:
        json.dump(json.loads(output_data[0].replace('```', '').replace('json', '')), f, ensure_ascii=False, indent=4)

100%|██████████| 10/10 [03:23<00:00, 20.32s/it]

CPU times: user 2min 58s, sys: 25.2 s, total: 3min 23s
Wall time: 3min 23s


In [38]:
!zip -r ./output.zip ./output

updating: output/ (stored 0%)
updating: output/v7.json (deflated 15%)
updating: output/v3.json (deflated 18%)
updating: output/v10.json (deflated 15%)
  adding: output/v5.json (deflated 16%)
  adding: output/v8.json (deflated 15%)
  adding: output/v4.json (deflated 16%)
  adding: output/v1.json (deflated 16%)
  adding: output/v6.json (deflated 15%)
  adding: output/V9.json (deflated 16%)
  adding: output/v2.json (deflated 15%)
